<a href="https://colab.research.google.com/github/Adityagujjar-08/dataviz-exercises-adityapatil/blob/main/lecture03_exercise_Aditya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [3]:
# Filter for Asian countries
asian_df = df[df['Region'] == 'Asia'].copy()
asian_countries = asian_df['Country'].unique()

fig = go.Figure()

# Design constants
HIGHLIGHT_COUNTRY = 'China'
HIGHLIGHT_COLOR = '#E63946'  # Crimson Red
GREY_COLOR = '#EEEEEE'       # Light background grey for context

# Group and plot each country's trajectory
for country in asian_countries:
    country_data = asian_df[asian_df['Country'] == country].sort_values('Year')
    last_point = country_data.iloc[-1]

    is_highlighted = (country == HIGHLIGHT_COUNTRY)
    line_color = HIGHLIGHT_COLOR if is_highlighted else GREY_COLOR
    line_width = 3 if is_highlighted else 1.5

    # Plot country line
    fig.add_trace(go.Scatter(
        x=country_data['Year'],
        y=country_data['CO2_Mt'],
        mode='lines',
        name=country,
        line=dict(color=line_color, width=line_width),
        hoverinfo='name+x+y'
    ))

    # Configure label styling
    label_text = f"<b>{country}</b>" if is_highlighted else country
    label_color = HIGHLIGHT_COLOR if is_highlighted else '#7F8C8D'
    label_size = 13 if is_highlighted else 11

    # Add direct label annotation at the end of the line
    fig.add_annotation(
        x=last_point['Year'],
        y=last_point['CO2_Mt'],
        text=label_text,
        xanchor='left',
        yanchor='middle',
        showarrow=False,
        font=dict(color=label_color, size=label_size),
        xshift=8
    )

# Set professional layout following minimalist design rules
fig.update_layout(
    title=dict(
        text="<b>China's CO2 Emissions Skyrocketed Over the Last Two Decades</b><br>" +
             "<span style='font-size: 13px; color: #7F8C8D;'>Annual CO2 emissions in megatonnes (Mt) for Asian countries (2000–2022)</span>",
        font=dict(size=18, color='#2C3E50'),
        x=0.04,
        y=0.92
    ),
    xaxis=dict(
        title="Year",
        showgrid=False,
        linecolor='#BDC3C7',
        ticks='outside',
        tickmode='linear',
        tick0=2000,
        dtick=5,
        range=[2000, 2026] # Extra padding on right side for the country labels
    ),
    yaxis=dict(
        title="CO2 Emissions (Mt)",
        showgrid=True,
        gridcolor='#F2F4F4', # Very soft grids to reduce visual noise
        linecolor='#BDC3C7'
    ),
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=120, t=100, b=60),
    width=850,
    height=550
)

fig.show()